# DRACO Lite through ScreamingFace and URL4 · fully explained

This is the architecture companion to `05_draco_quickstart.ipynb`. It constructs the same **7 solo
+ 9 Fusion** comparison and explains what is local, what becomes URL4, what the ScreamingFace
engine executes, and what returns to the SDK.

DRACO Lite is the production topology at miniature scale: one pinned real case, ten deterministic
criteria spanning all four rubric sections, and one judge pass. The available OpenRouter lineup
substitutes for the historical provider mix, so it demonstrates the protocol rather than
reproducing published scores.

```text
ScreamingFace SDK
  └─ one GET /v1?q=<complete candidate-study URL4>
       └─ ScreamingFace engine (Url4Node + versioned SF routes)
            ├─ pinned case + stable slice
            ├─ shared candidate DAG → model routes → AI Gateway → OpenRouter
            ├─ benchmark policy → managed web_search/web_fetch
            ├─ 9 tool-free synthesis calls
            ├─ final candidate answers → DRACO rubric judge
            └─ candidate mean aggregator → plaintext JSON StudyReport
```

The SDK never calls AI Gateway, OpenRouter, Tavily, Hugging Face, or model providers directly.

## 0 · Start, connect, and inspect the engine

Use a compatible engine whose deployment has an accepted Hugging Face dataset token and the
versioned DRACO routes. Connect OpenRouter in the panel. Benchmark and model discovery come from
the configured engine's registry—there is no client-side fallback catalog or bundled engine.

In [ ]:
import json
from pprint import pprint
from urllib.parse import urlencode

import screamingface as sf

sf.connect()

In [ ]:
print("Benchmarks:", sf.benchmarks.list(query="draco"))
print("OpenRouter models with research tools:")
pprint(sf.models.list(query="openrouter/", tools=("web_search",)))

## 1 · Build the complete candidate topology

`sf.Model` is one atomic answer Recipe. `sf.Fusion` combines other Recipes through a reducer.
Object identity matters: reusing `gpt` shares one answer node across every dependent Fusion; the two
fresh Opus objects in self-fusion remain two independent samples even though their routes match.

In [ ]:
DRACO_ANSWER_PROMPT = """You are answering a research-quality prompt.
Provide a thorough, well-reasoned answer in prose. Address every aspect the prompt raises.
Use clear structure
(headings, bullet lists where appropriate) and cite specific facts, methodologies, or sources
where relevant.

Do not refuse, abstain, or claim uncertainty unless the question is genuinely ambiguous — the
goal is to demonstrate depth of understanding. Length: aim for the level of detail the question
warrants; brevity that skips key points will be penalised by the rubric."""

DRACO_SYNTHESIS_PROMPT = """You are synthesising a single, comprehensive answer to a
research-quality prompt by combining N independent answers from a panel of models. The downstream
grader will score your output against a STRUCTURED RUBRIC of weighted criteria — your goal is to
maximise rubric coverage.

Procedure:
1. Read every panel answer carefully.
2. Identify which claims, facts, citations, or arguments each panel member contributes that the
   others miss.
3. Produce ONE unified prose response that combines the strongest reasoning, preserves specifics,
   resolves disagreements in favour of the better-supported claim, and uses clear structure.
4. Do not introduce new facts that no panel member provided.
5. Do not hedge or refuse.

Output: the unified prose answer, no preamble, no JSON wrapper."""

fable = sf.Model(
    "openrouter/anthropic/claude-fable-5",
    prompt=DRACO_ANSWER_PROMPT,
    params={"temperature": 0, "max_tokens": 8192},
)
opus = sf.Model(
    "openrouter/anthropic/claude-opus-4.8",
    prompt=DRACO_ANSWER_PROMPT,
    params={"temperature": 0, "max_tokens": 8192},
)
gpt = sf.Model(
    "openrouter/openai/gpt-5.5",
    prompt=DRACO_ANSWER_PROMPT,
    params={"temperature": 0, "max_tokens": 8192},
)
gemini_pro = sf.Model(
    "openrouter/google/gemini-3.1-pro-preview",
    prompt=DRACO_ANSWER_PROMPT,
    params={"temperature": 0, "max_tokens": 8192},
)
gemini_flash = sf.Model(
    "openrouter/google/gemini-3-flash-preview",
    prompt=DRACO_ANSWER_PROMPT,
    params={"temperature": 0, "max_tokens": 8192},
)
kimi = sf.Model(
    "openrouter/moonshotai/kimi-k2.5",
    prompt=DRACO_ANSWER_PROMPT,
    params={"temperature": 0, "max_tokens": 8192},
)
deepseek = sf.Model(
    "openrouter/deepseek/deepseek-v4-pro",
    prompt=DRACO_ANSWER_PROMPT,
    params={"temperature": 0, "max_tokens": 8192},
)
qwen = sf.Model(  # Fusion-only leaf
    "openrouter/qwen/qwen3.6-plus",
    prompt=DRACO_ANSWER_PROMPT,
    params={"temperature": 0, "max_tokens": 8192},
)

fable_plus_gpt = sf.Fusion(
    "fable-plus-gpt",
    members=[fable, gpt],
    reducer=sf.reducers.Model(
        model="openrouter/anthropic/claude-opus-4.8",
        prompt=DRACO_SYNTHESIS_PROMPT,
        params={"temperature": 0, "max_tokens": 8192},
    ),
)
frontier_trio = sf.Fusion(
    "frontier-trio",
    members=[opus, gpt, gemini_pro],
    reducer=sf.reducers.Model(
        model="openrouter/anthropic/claude-opus-4.8",
        prompt=DRACO_SYNTHESIS_PROMPT,
        params={"temperature": 0, "max_tokens": 8192},
    ),
)
opus_plus_gpt = sf.Fusion(
    "opus-plus-gpt",
    members=[opus, gpt],
    reducer=sf.reducers.Model(
        model="openrouter/anthropic/claude-opus-4.8",
        prompt=DRACO_SYNTHESIS_PROMPT,
        params={"temperature": 0, "max_tokens": 8192},
    ),
)
opus_self_fusion = sf.Fusion(
    "opus-self-fusion",
    members=[
        sf.Model(
            "openrouter/anthropic/claude-opus-4.8",
            name="opus-sample-1",
            prompt=DRACO_ANSWER_PROMPT,
            params={"temperature": 0.7, "max_tokens": 8192},
        ),
        sf.Model(
            "openrouter/anthropic/claude-opus-4.8",
            name="opus-sample-2",
            prompt=DRACO_ANSWER_PROMPT,
            params={"temperature": 0.7, "max_tokens": 8192},
        ),
    ],
    reducer=sf.reducers.Model(
        model="openrouter/anthropic/claude-opus-4.8",
        prompt=DRACO_SYNTHESIS_PROMPT,
        params={"temperature": 0, "max_tokens": 8192},
    ),
)
budget_trio = sf.Fusion(
    "budget-trio",
    members=[gemini_flash, kimi, deepseek],
    reducer=sf.reducers.Model(
        model="openrouter/anthropic/claude-opus-4.8",
        prompt=DRACO_SYNTHESIS_PROMPT,
        params={"temperature": 0, "max_tokens": 8192},
    ),
)
beat_runner_up = sf.Fusion(
    "beat-runner-up",
    members=[opus, gpt, deepseek],
    reducer=sf.reducers.Model(
        model="openrouter/anthropic/claude-opus-4.8",
        prompt=DRACO_SYNTHESIS_PROMPT,
        params={"temperature": 0, "max_tokens": 8192},
    ),
)
pareto_cross = sf.Fusion(
    "pareto-cross",
    members=[deepseek, kimi, gpt],
    reducer=sf.reducers.Model(
        model="openrouter/deepseek/deepseek-v4-pro",
        prompt=DRACO_SYNTHESIS_PROMPT,
        params={"temperature": 0, "max_tokens": 8192},
    ),
)
pareto_lean = sf.Fusion(
    "pareto-lean",
    members=[deepseek, kimi],
    reducer=sf.reducers.Model(
        model="openrouter/deepseek/deepseek-v4-pro",
        prompt=DRACO_SYNTHESIS_PROMPT,
        params={"temperature": 0, "max_tokens": 8192},
    ),
)
best_open_source = sf.Fusion(
    "best-open-source",
    members=[deepseek, kimi, qwen],
    reducer=sf.reducers.Model(
        model="openrouter/deepseek/deepseek-v4-pro",
        prompt=DRACO_SYNTHESIS_PROMPT,
        params={"temperature": 0, "max_tokens": 8192},
    ),
)

candidates = (
    fable,
    opus,
    gpt,
    gemini_pro,
    gemini_flash,
    kimi,
    deepseek,
    fable_plus_gpt,
    frontier_trio,
    opus_plus_gpt,
    opus_self_fusion,
    budget_trio,
    beat_runner_up,
    pareto_cross,
    pareto_lean,
    best_open_source,
)

len(candidates)

The sixteen roots are seven solo Models followed by nine Fusions. Qwen is a
Fusion-only leaf. Across the whole graph there are ten distinct researched model nodes: eight named
leaves plus two independent Opus samples.

## 2 · Inspect reusable answer Recipes

Each candidate can still be shared as its parameterized answer Recipe. It contains `$question` and
does not include a dataset, slice, grader, or aggregator.

In [ ]:
for candidate in candidates:
    print(f"\n--- {candidate.name} ---\n{candidate.url4}")

These answer-level URL4s are useful for inspection and reuse. They are not the
benchmark result recipe. The complete reproducible study is compiled by the loaded benchmark.

## 3 · Load the benchmark manifest

In [ ]:
draco = sf.benchmarks.load("draco-lite@1")

{
    "id": draco.id,
    "title": draco.title,
    "grader": {
        "kind": draco.grader.kind,
        "model": draco.grader.model,
        "passes": draco.grader.passes,
        "params": draco.grader.params,
    },
    "aggregator": draco.aggregator.kind,
    "tools": [tool.id for tool in draco.tools],
    "max_tool_calls": draco.max_tool_calls,
}

`load` validates the engine manifest; it does not download cases. During execution,
the versioned case route loads the pinned `perplexity-ai/draco` revision with the engine's
`HF_TOKEN`. DRACO Lite selects the first case, seals the first positive and negative criteria, then
adds the first criterion from each remaining rubric section, then fills in dataset order, for ten
total. Its grader uses one
pass; production `draco@1` uses complete rubrics and five.

The deployment itself is authored from the same compact SDK values available to researchers:

```python
sf.Benchmark(
    "my-research@1",
    cases=[sf.Case("q1", "Question", reference={...})],
    grader=sf.graders.Rubric(model="...", prompt="...", passes=1, params={...}),
    aggregator=sf.aggregators.Mean(),
    tools=(sf.tools.WebSearch(max_results=5), sf.tools.WebFetch()),
    max_tool_calls=12,
)
```

This is benchmark authoring, not an ETL DSL. A deployed benchmark exposes immutable versioned
routes and a small manifest.

## 4 · Compile the one complete study URL4 without running it

In [ ]:
study_url4 = draco.url4(candidates)
print(study_url4)

In [ ]:
request_target = "/v1?" + urlencode({"q": study_url4})
print("GET", request_target[:700] + "…")

This is one top-level URL4 transaction—not sixteen requests. Its readable shape is:

```text
/benchmarks/draco-lite/1/cases*(
  tool_policy = /benchmarks/draco/1/tool-policy,
  candidate_spec = {
    nodes: { shared Models and Fusions },
    candidates: { 16 ordered names → root node IDs }
  },
  candidate_input = { case, question, rubric, tool policy },
  case_result = /benchmarks/draco-lite/1/evaluate-candidates(
    candidate_spec
  )!candidate_input
)!case_result;
iteration.slice=0:1;
iteration.on_error=collect
  !/aggregators/candidate-mean/1()!'Aggregate candidate benchmark results'
```

The candidate specification is data inside ordinary URL4 composition. The versioned engine route
executes that declared DAG with shared-node memoization and per-root failure isolation. No URL4 SDK
or AI Gateway modification is required.

## 5 · What executes inside the candidate route

For the single case:

1. The engine starts the ten distinct research leaves concurrently.
2. Every leaf receives the same benchmark-owned `web_search`/`web_fetch` policy and 12-call budget.
3. Each Fusion waits only for its member nodes, then makes one tool-free synthesis call.
4. A failed node fails only candidates that depend on it; unrelated candidate roots continue.
5. The engine grades only the sixteen final candidate answers—not their members again.
6. The candidate aggregator preserves declaration order, coverage, scores, and typed failures.

The nominal model-call count is therefore:

```text
10 researched samples
 9 synthesis calls
160 judge calls = 16 candidates × 10 criteria × 1 pass
──
179 model calls
```

Provider-managed search operations and explicit judge-output validation retries are additional.

## 6 · Tools and ownership

The URL4 carries provider-neutral capability policy, never Tavily credentials or
provider-specific tool code. OpenRouter routes map `web_search` and `web_fetch` to OpenRouter's
managed server tools. Verified bare Hugging Face routes map the same policy to the ScreamingFace
engine's bounded Tavily agent loop. Synthesis and judging are tool-free.

AI Gateway remains model transport and model-credential storage. The ScreamingFace engine owns
benchmark policy, tool-backend selection, candidate orchestration, and grading.

## 7 · Grading semantics

For each final answer, the grader makes one request per criterion and pass. Each
judge sees the original question, candidate answer, one criterion, and whether it is positive or
negative. It never sees the criterion weight or sibling criteria and must return
`{explanation, criterion_status: MET|UNMET}`.

```text
normalized_score = clamp(Σ(MET × weight) / Σ(positive weights), 0, 1)
```

Positive MET adds reward; negative MET subtracts a penalty. Missing verdicts reduce coverage rather
than silently becoming zero.

## 8 · Plaintext engine response

In [ ]:
example_response = {
    "schema": "screamingface.study-report.v1",
    "benchmark_id": "draco-lite@1",
    "case_ids": ["<pinned case UUID>"],
    "candidates": {
        "claude-fable-5": {
            "n_cases": 1,
            "n_scored": 1,
            "coverage": 1.0,
            "score": 0.5,
            "metrics": {"normalized_score": 0.5, "verdict_coverage": 1.0},
            "failures": [],
            "complete": True,
        },
        "frontier-trio": {
            "n_cases": 1,
            "n_scored": 1,
            "coverage": 1.0,
            "score": 1.0,
            "metrics": {"normalized_score": 1.0, "verdict_coverage": 1.0},
            "failures": [],
            "complete": True,
        },
    },
    "complete": True,
}

print(json.dumps(example_response, indent=2))

The URL4 engine still returns `text/plain`; its body is strict JSON. The SDK
validates
the schema, benchmark and case identities, ordered candidate set, score ranges, coverage, metrics,
and typed failures before constructing immutable `sf.StudyReport` and `sf.CandidateReport` values.

## 9 · Execute deliberately

In [ ]:
# This performs the real paid 83-call nominal run:
# report = draco.evaluate(candidates)
# report

study_url4

When executed, `report.url4` is exactly `study_url4`. `report.candidates` contains
all
sixteen independently scored roots and `report.best` selects the highest scored completed one.

Switching to production DRACO is intentionally more than raising a number: `draco@1` changes to
100 cases, complete rubrics, and five judge passes. Published-result parity also requires the
historical pinned model/provider behavior and an audited full run.